# Recolección de telemetría de tráfico

Usá esta notebook cuando necesites convertir un video de tránsito en **telemetría cruda por minuto**. No entrena ni clasifica estados: observa vehículos, estima velocidades y deja datos listos para el entrenamiento.

| Necesitás | La notebook entrega | Requisito mínimo |
|---|---|---|
| Un MP4 de tránsito | Video anotado + CSV raw | 60 segundos para generar una fila |
| PostgreSQL, sólo si querés persistir | Filas idempotentes en `vaaet_raw.traffic_data` | Perfil `collection` |

**Inicio rápido recomendado:** dejá `PERSIST_TO_DATABASE=False` y `HUD_DEBUG=False`, ejecutá `Run All` y subí el video cuando aparezca el selector.

```text
Video → detección y seguimiento → video anotado
                              └→ telemetría cruda por minuto
                                  ├→ CSV
                                  └→ PostgreSQL opcional
```

<details>
<summary><strong>Ver las tres recetas de configuración</strong></summary>

### A. Recolección local recomendada
```python
PERSIST_TO_DATABASE = False
HUD_DEBUG = False
```
Genera el video anotado y acumula el CSV. No necesita Secrets.

### B. Recolección con PostgreSQL
```python
PERSIST_TO_DATABASE = True
HUD_DEBUG = False
```
Además del video y el CSV, inserta telemetría en `vaaet_raw`. Necesita el perfil `collection`.

### C. Diagnóstico técnico
```python
PERSIST_TO_DATABASE = False
HUD_DEBUG = True
```
Agrega IDs y señales técnicas al HUD. Usalo para revisar tracking, no para presentar el video al público.

</details>

Los nombres `bridge_YYYY-MM-DD_HH-MM-SS_to_HH-MM-SS.mp4` conservan la hora real. Con un nombre libre se usa la hora de procesamiento y la trazabilidad es menor.

In [ ]:
# Workflow configuration — edit only this cell
PERSIST_TO_DATABASE = False
HUD_DEBUG = False

print("✅ Configuración validada")
print("🧭 Flujo seleccionado: recolección de telemetría")
print(f"   PostgreSQL: {'activado; se usará el perfil collection' if PERSIST_TO_DATABASE else 'desactivado'}")
print(f"   HUD: {'diagnóstico técnico' if HUD_DEBUG else 'público y simplificado'}")
print("   Salidas: video anotado + CSV raw" + (" + PostgreSQL" if PERSIST_TO_DATABASE else ""))
print("➡️ Siguiente paso: ejecutá la preparación del entorno.")

In [ ]:
# Environment setup — run once per Colab runtime
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_URL = "https://github.com/zgfnicolas/vaaet.git"
WORKSPACE_DIR = Path("/content/vaaet"); ML_ROOT = WORKSPACE_DIR / "vaaet-ml"
if IN_COLAB:
    if (WORKSPACE_DIR / ".git").is_dir():
        subprocess.check_call(["git", "-C", str(WORKSPACE_DIR), "pull", "--ff-only"])
    else:
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(WORKSPACE_DIR)])
    REPO_ROOT = ML_ROOT.resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((path for path in candidates if (path / "pyproject.toml").is_file() and (path / "src/vaaet").is_dir()), None)
    if REPO_ROOT is None:
        REPO_ROOT = next((path / "vaaet-ml" for path in candidates if (path / "vaaet-ml/pyproject.toml").is_file()), None)
    if REPO_ROOT is None:
        raise RuntimeError("No se encontró el componente vaaet-ml.")
os.chdir(REPO_ROOT)
WORKFLOW_EXTRAS = "vision,database"
project_requirement = f"{REPO_ROOT}[{WORKFLOW_EXTRAS}]"
install_command = [sys.executable, "-m", "pip", "install", "-q"]
if IN_COLAB:
    install_command.append(project_requirement)
else:
    install_command.extend(["-e", project_requirement])
installation = subprocess.run(install_command, capture_output=True, text=True, check=False)
if installation.returncode:
    print(installation.stdout.strip() or "(empty)")
    print(installation.stderr.strip() or "(empty)")
    raise RuntimeError(f"VAAET installation failed with extras={WORKFLOW_EXTRAS}.")
for module_name in tuple(sys.modules):
    if module_name == "vaaet" or module_name.startswith("vaaet."):
        sys.modules.pop(module_name, None)
from vaaet.runtime import bootstrap_notebook_runtime
RUNTIME = bootstrap_notebook_runtime(
    workspace_root=WORKSPACE_DIR if IN_COLAB else REPO_ROOT.parent,
    ml_root=REPO_ROOT,
    in_colab=IN_COLAB,
    framework="torch",
    require_gpu=IN_COLAB,
)
VAAET_PACKAGE_FILE = RUNTIME.package_file
GIT_COMMIT = RUNTIME.git_commit


In [ ]:
# Workflow imports — do not edit
import cv2
import numpy as np
import pandas as pd
import psycopg2
import sqlalchemy
import torch
import ultralytics

from vaaet.data.database import DatabaseProfile, database_engine, get_optional_database_settings, inspect_database
from vaaet.data.datasets import merge_raw_telemetry_csv
from vaaet.data.pipeline_runs import PipelineRunMetadata, PipelineWorkflow, pipeline_run
from vaaet.data.persistence import persist_raw_telemetry
from vaaet.logging import configure_logging
from vaaet.workflow_config import CollectionWorkflowConfig
from vaaet.vision.analysis import analyze_video
from vaaet.vision.hud import HudConfig

configure_logging()
WORKFLOW_CONFIG = CollectionWorkflowConfig(persist_to_database=PERSIST_TO_DATABASE, hud_debug=HUD_DEBUG)


## 1. Seleccionar el video

En Colab se abrirá el selector de archivos. En local, colocá un ejemplo no sensible en `data/sample/` o asigná una ruta absoluta a `VIDEO_PATH`.

Ejemplos: 16 segundos producen sólo video; 60 segundos producen una fila; 125 segundos producen dos filas y descartan los 5 segundos finales.

In [ ]:
VIDEO_PATH: Path | None = None
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded))).resolve()
else:
    candidate = REPO_ROOT / "data/sample/sample.mp4"
    VIDEO_PATH = candidate if candidate.is_file() else None

print(f"Video seleccionado: {VIDEO_PATH or 'ninguno'}")
print("➡️ Siguiente paso: procesá el video." if VIDEO_PATH else "⚠️ Subí o seleccioná un MP4 para continuar.")

## 2. Procesar el video

YOLO descarga sus pesos oficiales la primera vez; no se guardan en Git. Durante el proceso verás detecciones, velocidades aproximadas y conteos acumulados.

Al terminar, la celda muestra cuántos minutos completos generó, cuánto tiempo parcial descartó y dónde quedaron el video y el CSV.

In [ ]:
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Seleccioná o subí un MP4 válido antes de continuar.")

OUTPUT_VIDEO = Path("/content") / f"{VIDEO_PATH.stem}_vaaet_annotated.mp4" if IN_COLAB else VIDEO_PATH.with_name(f"{VIDEO_PATH.stem}_vaaet_annotated.mp4")
_run_metadata = PipelineRunMetadata(workflow=PipelineWorkflow.COLLECTION, git_commit=GIT_COMMIT, source_kind="video", clip_id=VIDEO_PATH.stem)
with pipeline_run(_run_metadata, local_manifest_directory=REPO_ROOT / "data/processed/pipeline-runs") as _run:
    result = analyze_video(
        VIDEO_PATH, OUTPUT_VIDEO, hud_config=HudConfig(debug=HUD_DEBUG)
    )
    _run.set_output_rows(len(result.telemetry))
COLLECTION_PIPELINE_RUN_ID = str(_run.id)

RAW_CSV = REPO_ROOT / "data/raw/traffic_data_raw.csv"
df_raw = None
display(result.telemetry)
print(f"✅ Video anotado: {result.video_path}")
if result.telemetry.empty:
    print("ℹ️ El video se procesó bien, pero no contiene un minuto completo.")
    print("   No se generará telemetría, no se actualizará el CSV y PostgreSQL se omitirá.")
    print(f"   Duración procesada: {result.processed_duration_seconds:.1f}s | mínimo: 60.0s")
else:
    df_raw = merge_raw_telemetry_csv(result.telemetry, RAW_CSV)
    print(f"✅ Minutos completos: {result.complete_minutes} | tramo final descartado: {result.discarded_partial_seconds:.1f}s")
    print(f"✅ CSV canónico: {RAW_CSV} ({len(df_raw)} filas únicas)")
print("➡️ Siguiente paso: revisá la persistencia opcional.")

if IN_COLAB:
    from google.colab import files

    files.download(str(result.video_path))
    if not result.telemetry.empty and RAW_CSV.is_file():
        files.download(str(RAW_CSV))

## 3. Guardar en PostgreSQL (opcional)

Esta etapa se omite por defecto. Si la activaste, usa únicamente el perfil `collection` y escribe en `vaaet_raw`. Repetir el mismo clip no duplica filas porque `(clip_id, record_time)` identifica cada minuto.

Las credenciales se configuran según la [guía canónica de Colab](../../../docs/operations/colab-guide.md#secrets-y-postgresql); tenerlas disponibles nunca activa la escritura por sí solo.

In [ ]:
if result.telemetry.empty:
    print("ℹ️ PostgreSQL omitido: el video no tiene un minuto completo.")
elif PERSIST_TO_DATABASE:
    settings = get_optional_database_settings(DatabaseProfile.COLLECTION)
    if settings is None:
        raise RuntimeError("Activaste PostgreSQL, pero el perfil collection no está configurado.")
    with database_engine(settings) as db_engine:
        health = inspect_database(db_engine, DatabaseProfile.COLLECTION)
        print(f"PostgreSQL {health.server_version} | TLS={health.ssl_enabled} | schemas={health.available_schemas}")
        inserted = persist_raw_telemetry(result.telemetry, engine=db_engine)
    print(f"✅ PostgreSQL: {inserted} filas nuevas en vaaet_raw.traffic_data")
else:
    print("ℹ️ PostgreSQL desactivado; el video y el CSV siguen disponibles.")
print("✅ Flujo de recolección terminado.")